In [3]:
import random
from collections import defaultdict
import tkinter as tk
from tkinter import simpledialog, messagebox

# Node class: Simulate a single node in PBFT system
class Node:
    def __init__(self, node_id, is_malicious=False):
        self.node_id = node_id
        self.is_malicious = is_malicious  # Whether the node is malicious
        self.preprepare_val = None        # Pre-prepare message (always correct value)
        self.prepare_cnt = defaultdict(int)  # Count of prepare messages received
        self.commit_cnt = defaultdict(int)   # Count of commit messages received
        self.prepared = False               # Whether reach prepared state
        self.committed = False              # Whether reach committed state
        self.result = None                  # Final consensus result

# PBFT Simulator class: Control the entire consensus process
class PBFT_Simulator:
    def __init__(self, total_nodes, malicious_num):
        self.N = total_nodes                # Total number of nodes
        self.f = malicious_num              # Number of malicious nodes
        self.honest_num = total_nodes - malicious_num  # Number of honest nodes
        
        # Standard PBFT security condition: f ≤ (N-1)/3
        self.safe = (self.f <= (self.N - 1) // 3)
        
        # Consensus threshold: more than 2/3 of total nodes (rounded up)
        self.threshold = (2 * self.N) // 3 + 1

        # Create nodes: Node 0 is primary node (always honest)
        # Randomly select f nodes from 1~N-1 as malicious nodes
        malicious_ids = random.sample([i for i in range(1, self.N)], self.f) if self.N > 1 else []
        self.nodes = []
        self.honest_node_ids = []  # Store all honest node IDs
        for i in range(self.N):
            is_mal = (i in malicious_ids)
            self.nodes.append(Node(i, is_malicious=is_mal))
            if not is_mal:
                self.honest_node_ids.append(i)

        self.value = "block_hash_123456"  # Correct proposal value

    # 1. Pre-prepare phase: All nodes (including malicious) receive correct value
    def pre_prepare(self):
        print("\n===== Pre-prepare Phase (All nodes receive correct proposal) =====")
        for node in self.nodes:
            node.preprepare_val = self.value  # Key rule: All nodes get correct value!
            status = "Malicious" if node.is_malicious else "Honest"
            print(f"Node {node.node_id} [{status}] received: {node.preprepare_val}")

    # 2. Prepare phase: Malicious nodes broadcast fake messages
    def prepare(self):
        print("\n===== Prepare Phase (Malicious nodes broadcast fake messages) =====")
        # Step 1: All nodes broadcast messages
        for sender in self.nodes:
            # Determine broadcast content
            if sender.is_malicious:
                # Malicious node: broadcast fake message
                send_msg = f"fake_{self.value}_{sender.node_id}"
            else:
                # Honest node: broadcast real message
                send_msg = self.value

            # Broadcast to all nodes
            for receiver in self.nodes:
                if receiver.is_malicious:
                    continue  # Malicious nodes don't count messages
                receiver.prepare_cnt[send_msg] += 1  # Receiver counts messages

        # Step 2: Show honest node info and check Prepared state
        print("\n[Prepare Phase Statistics (Honest Nodes Only)]:")
        print(f"Total honest nodes: {self.honest_num}")
        print(f"Honest node ID list: {self.honest_node_ids}")
        print("\nPrepared state check for each honest node:")
        for node in self.nodes:
            if node.is_malicious:
                continue
            # Count correct messages
            correct_cnt = node.prepare_cnt.get(self.value, 0)
            # Check if reach threshold
            if correct_cnt >= self.threshold:
                node.prepared = True
                print(f"→ Node {node.node_id}: Correct messages={correct_cnt} ≥ threshold({self.threshold}), Prepared state achieved")
            else:
                node.prepared = False
                print(f"→ Node {node.node_id}: Correct messages={correct_cnt} < threshold({self.threshold}), Prepared state not achieved")

    # 3. Commit phase: Only prepared honest nodes broadcast commit messages
    def commit(self):
        print("\n===== Commit Phase =====")
        # Step 1: Broadcast commit messages
        for sender in self.nodes:
            if sender.is_malicious or not sender.prepared:
                continue  # Malicious/unprepared nodes don't broadcast
            # Broadcast correct proposal to all honest nodes
            for receiver in self.nodes:
                if receiver.is_malicious:
                    continue
                receiver.commit_cnt[self.value] += 1

        # Step 2: Check if each honest node reaches committed state
        print("\n[Commit Phase Statistics (Honest Nodes Only)]:")
        for node in self.nodes:
            if node.is_malicious:
                continue
            commit_cnt = node.commit_cnt.get(self.value, 0)
            print(f"Node {node.node_id}: Received commit messages={commit_cnt}")
            if commit_cnt >= self.threshold:
                node.committed = True
                node.result = self.value
            else:
                node.committed = False
                node.result = None

    # Run the complete PBFT consensus process
    def run(self):
        print("===== PBFT Consensus Simulation Started =====")
        print(f"Total nodes: {self.N} | Malicious nodes: {self.f} | Honest nodes: {self.honest_num}")
        print(f"PBFT Security Threshold: f ≤ (N-1)/3 → Current {self.f} ≤ {(self.N-1)//3} → {self.safe}")
        print(f"Consensus Threshold (≥2/3 of total nodes): {self.threshold}")

        self.pre_prepare()
        self.prepare()
        self.commit()
        self.show_result()

    # Show final consensus result
    def show_result(self):
        print("\n===== Final Consensus Result ======")
        honest_results = []
        for node in self.nodes:
            if node.is_malicious:
                continue
            print(f"Honest Node {node.node_id} Final Result: {node.result}")
            honest_results.append(node.result)

        # Judge if consensus succeeded
        if len(set(honest_results)) == 1 and honest_results[0] is not None:
            print("\n✅ Consensus Success! All honest nodes reached agreement")
        else:
            print("\n❌ Consensus Failed! Honest nodes did not reach agreement")

# Get user input via GUI dialog
def get_input():
    # Initialize tkinter and hide main window
    root = tk.Tk()
    root.withdraw()

    # Get total node count (must be ≥4)
    while True:
        total_input = simpledialog.askstring("Input Total Nodes", "Please enter total number of PBFT nodes (positive integer ≥4)：", parent=root)
        if total_input is None:  # User cancelled input
            exit()
        # Validate input
        try:
            total_nodes = int(total_input)
            if total_nodes >= 4:
                break
            else:
                messagebox.showerror("Input Error", "Total nodes must be ≥4!")
        except ValueError:
            messagebox.showerror("Input Error", "Please enter a valid positive integer!")

    # Get malicious node count (0 ≤ malicious nodes < total nodes)
    while True:
        malicious_input = simpledialog.askstring("Input Malicious Nodes", f"Total nodes={total_nodes}, please enter number of malicious nodes (0 ≤ number < {total_nodes})：", parent=root)
        if malicious_input is None:  # User cancelled input
            exit()
        # Validate input
        try:
            malicious_count = int(malicious_input)
            if 0 <= malicious_count < total_nodes:
                break
            else:
                messagebox.showerror("Input Error", f"Malicious nodes must satisfy: 0 ≤ number < {total_nodes}!")
        except ValueError:
            messagebox.showerror("Input Error", "Please enter a valid integer!")
    
    return total_nodes, malicious_count

# Main program entry
if __name__ == "__main__":
    # Get user input
    N, F = get_input()
    # Initialize simulator and run
    sim = PBFT_Simulator(N, F)
    sim.run()

===== PBFT Consensus Simulation Started =====
Total nodes: 10 | Malicious nodes: 3 | Honest nodes: 7
PBFT Security Threshold: f ≤ (N-1)/3 → Current 3 ≤ 3 → True
Consensus Threshold (≥2/3 of total nodes): 7

===== Pre-prepare Phase (All nodes receive correct proposal) =====
Node 0 [Honest] received: block_hash_123456
Node 1 [Malicious] received: block_hash_123456
Node 2 [Malicious] received: block_hash_123456
Node 3 [Honest] received: block_hash_123456
Node 4 [Honest] received: block_hash_123456
Node 5 [Malicious] received: block_hash_123456
Node 6 [Honest] received: block_hash_123456
Node 7 [Honest] received: block_hash_123456
Node 8 [Honest] received: block_hash_123456
Node 9 [Honest] received: block_hash_123456

===== Prepare Phase (Malicious nodes broadcast fake messages) =====

[Prepare Phase Statistics (Honest Nodes Only)]:
Total honest nodes: 7
Honest node ID list: [0, 3, 4, 6, 7, 8, 9]

Prepared state check for each honest node:
→ Node 0: Correct messages=7 ≥ threshold(7), Prepa

In [4]:
# Main program entry
if __name__ == "__main__":
    # Get user input
    N, F = get_input()
    # Initialize simulator and run
    sim = PBFT_Simulator(N, F)
    sim.run()

===== PBFT Consensus Simulation Started =====
Total nodes: 10 | Malicious nodes: 4 | Honest nodes: 6
PBFT Security Threshold: f ≤ (N-1)/3 → Current 4 ≤ 3 → False
Consensus Threshold (≥2/3 of total nodes): 7

===== Pre-prepare Phase (All nodes receive correct proposal) =====
Node 0 [Honest] received: block_hash_123456
Node 1 [Malicious] received: block_hash_123456
Node 2 [Malicious] received: block_hash_123456
Node 3 [Malicious] received: block_hash_123456
Node 4 [Malicious] received: block_hash_123456
Node 5 [Honest] received: block_hash_123456
Node 6 [Honest] received: block_hash_123456
Node 7 [Honest] received: block_hash_123456
Node 8 [Honest] received: block_hash_123456
Node 9 [Honest] received: block_hash_123456

===== Prepare Phase (Malicious nodes broadcast fake messages) =====

[Prepare Phase Statistics (Honest Nodes Only)]:
Total honest nodes: 6
Honest node ID list: [0, 5, 6, 7, 8, 9]

Prepared state check for each honest node:
→ Node 0: Correct messages=6 < threshold(7), Prep